# Impact of External Ressource on Detection Performance

In [1]:
import sys
import os
from os.path import join
import pandas as pd
import numpy as np
from itertools import combinations

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from config import DATA_DIR, EXP_COLS, CLASS_COLS
from utils.analysis_helpers import wrap_model_comparison, wrap_model_comparison_union, print_model_comparison_results, combine_model_comparison_outputs, extract_metric_values, compute_balanced_metrics

In [2]:
PROVIDER="llama"

In [3]:
df_b_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_1.feather"))
df_d_1 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_1.feather"))
df_b_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_bloomington_label_0.feather"))
df_d_0 = pd.read_feather(join(DATA_DIR,f"{PROVIDER}_decoding_label_0.feather"))

In [4]:
df_d_1.columns

Index(['comment_cleaned', 'id', 'discourse', 'comment_level',
       'comment_codes_all', 'source_outlet',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_tax',
       'explanation_tax', 'classification_tax_ex', 'explanation_tax_ex',
       'classification_no_kb_cleaned', 'comment_codes_all_list',
       'comment_codes_all_chapters', 'comment_codes_all_sections',
       'explanation_tax_chapters', 'explanation_tax_ex_chapters',
       'explanation_ihra_explanation_sections'],
      dtype='object')

In [5]:
df_b_1.columns

Index(['comment_cleaned', 'id', 'keyword', 'ihra_section_1', 'ihra_section_2',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_tax',
       'explanation_tax', 'classification_tax_ex', 'explanation_tax_ex',
       'classification_no_kb_cleaned', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters', 'explanation_ihra_explanation_sections'],
      dtype='object')

In [6]:
cols_d = ['comment_cleaned', 'discourse', 'comment_level', 
          'comment_codes_all', 'source_outlet'] + EXP_COLS + CLASS_COLS

for c in cols_d: 
    if c not in df_d_0.columns:
        print(c)

In [7]:
cols_b = ['comment_cleaned', 'keyword', 'ihra_section_1', 'ihra_section_2'] + EXP_COLS + CLASS_COLS

for c in cols_b: 
    if c not in df_b_0.columns:
        print(c)

In [8]:
N_neg_total_d = 39424
N_neg_total_b = 9358

### Merge positive and negative samples

In [9]:
df_d_0["label"] = 0
df_d_1["label"] = 1
df_b_0["label"] = 0
df_b_1["label"] = 1

cols_b.insert(0,"label")
cols_d.insert(0,"label")

In [10]:
df_b = pd.concat([df_b_0[cols_b], df_b_1[cols_b]], ignore_index=True)
df_d = pd.concat([df_d_0[cols_d], df_d_1[cols_d]], ignore_index=True)

In [11]:
df_b.label.value_counts()

label
1    1858
0    1000
Name: count, dtype: int64

In [12]:
df_d.label.value_counts()

label
1    2977
0     992
Name: count, dtype: int64

### Combine datasets

In [13]:
w_neg_d = N_neg_total_d / len(df_d_0)
w_neg_b = N_neg_total_b / len(df_b_0)

df_d["dataset_id"] = "d"
df_b["dataset_id"] = "b"

df_union = pd.concat([df_d, df_b], ignore_index=True)

N_neg_total_by_dataset = {
    "d": N_neg_total_d,
    "b": N_neg_total_b,
}

### Distribution of responses in the positive samples

In [14]:
# Bloomington
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_b_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.002691,NaN,NaN,NaN
No,0.517761,0.314316,0.248654,0.294403
Unsure,NaN,0.011841,0.018837,0.022605
Yes,0.479548,0.673843,0.732508,0.682992


In [15]:
# Decoding  
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_d_1[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.005039,0.000672,NaN,NaN
No,0.706416,0.432314,0.315082,0.394021
None,NaN,0.000336,NaN,NaN
Unsure,NaN,0.014780,0.021162,0.014444
Yes,0.288546,0.551898,0.663755,0.591535


### Distribution of responses in the negative samples

In [16]:
# Bloomington
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_b_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.001,NaN,NaN,NaN
No,0.950,0.870,0.827,0.856
Unsure,NaN,0.003,0.015,0.006
Yes,0.049,0.127,0.158,0.138


In [17]:
# Decoding
tmp = {}
for c in CLASS_COLS:
    tmp[c] = df_d_0[c].value_counts(normalize=True)
pd.DataFrame(tmp)

,classification_no_kb_cleaned,classification_ihra_explanation_cleaned,classification_tax,classification_tax_ex
More context needed,0.018145,0.007056,NaN,NaN
No,0.958669,0.922379,0.827621,0.907258
None,NaN,0.001008,0.001008,0.002016
Unsure,NaN,0.017137,0.071573,0.025202
Yes,0.023185,0.052419,0.099798,0.065524


### Compute metrics and CIs on weighted union of both datasets

- B and D have the same weight (0.5)
- CI for recall reflects sampling from population; Ci for precision and F1 reflects sampling from population and 1000er sampling from dataset, thus larger CIs

In [18]:
model_comparison_metrics = compute_balanced_metrics(df_B=df_b, df_D=df_d, N_neg_total_B=N_neg_total_b, N_neg_total_D=N_neg_total_d)
model_comparison_metrics

,model,metric,value,ci_low,ci_high
0,no_kb,precision,0.572348,0.517915,0.642614
1,no_kb,recall,0.384047,0.370050,0.398306
2,no_kb,f1,0.458622,0.436493,0.481300
3,ihra,precision,0.477962,0.440978,0.523490
4,ihra,recall,0.612870,0.599013,0.626616
5,ihra,f1,0.536983,0.511648,0.565086
6,tax,precision,0.406811,0.379131,0.440060
7,tax,recall,0.698132,0.684901,0.711438
8,tax,f1,0.512061,0.487887,0.538907
9,tax_ex,precision,0.450495,0.416770,0.491291


In [19]:
model_comparison_metrics.to_parquet(f"{PROVIDER}_model_comparison_metrics.parquet", index=False)

### Export errors for qualitative analysis

In [20]:
df_b[(df_b["label"]==0) & (df_b["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_lexicon"]=="Yes")].to_csv(join(DATA_DIR,"lexicon_false_positives_decoding.csv"), index=False)
df_b[(df_b["label"]==0) & (df_b["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_bloomington.csv"), index=False)
df_d[(df_d["label"]==0) & (df_d["classification_ihra_explanation_cleaned"]=="Yes")].to_csv(join(DATA_DIR,"ihra_exp_false_positives_decoding.csv"), index=False)

### Statistical model comparison vs baseline (no_kb)

## Bloomington

Mapping "Yes" to 1 and Unsure to 0

1. class proportion as in the original dataset
2. class balance

In [21]:
res_b_weighted = wrap_model_comparison(df=df_b, N_neg_total=N_neg_total_b)
print_model_comparison_results(res_b_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.5130
metric(no_kb): 0.6602
Diff (model - baseline): -0.1472
95% CI: [-0.2002, -0.0985]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.3002 (95% CI: [-0.4138, -0.1993])

Model: tax
metric(tax): 0.4793
metric(no_kb): 0.6602
Diff (model - baseline): -0.1809
95% CI: [-0.2365, -0.1284]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.3676 (95% CI: [-0.4871, -0.2593])

Model: tax_ex
metric(tax_ex): 0.4956
metric(no_kb): 0.6602
Diff (model - baseline): -0.1646
95% CI: [-0.2219, -0.1128]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.3350 (95% CI: [-0.4573, -0.2277])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.6738
metric(no_kb): 0.4795
Diff (model - baseline): 0.1943
95% CI: [nan, nan]
p-value (raw): 1.778e-105
p-value (adjusted): 3.555e-105
Effect size h: 0.3960 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.7325
metric(no_kb): 0.4795
Diff (model - baseline): 0.2530
95% CI: [nan, nan]
p-va

### Decoding

Mapping "Yes" to 1 and "Unsure" to 0

1. class proportion as in the original dataset
2. class balance

In [22]:
res_d_weighted = wrap_model_comparison(df=df_d, N_neg_total=N_neg_total_d)
print_model_comparison_results(res_d_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.4429
metric(no_kb): 0.4845
Diff (model - baseline): -0.0416
95% CI: [-0.1331, 0.0276]
p-value (raw): 0.2654
p-value (adjusted): 0.2654
Effect size h: -0.0834 (95% CI: [-0.2682, 0.0558])

Model: tax
metric(tax): 0.3343
metric(no_kb): 0.4845
Diff (model - baseline): -0.1501
95% CI: [-0.2538, -0.0721]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.3067 (95% CI: [-0.5144, -0.1492])

Model: tax_ex
metric(tax_ex): 0.4054
metric(no_kb): 0.4845
Diff (model - baseline): -0.0791
95% CI: [-0.1795, -0.0015]
p-value (raw): 0.0454
p-value (adjusted): 0.0908
Effect size h: -0.1594 (95% CI: [-0.3621, -0.0030])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.5519
metric(no_kb): 0.2885
Diff (model - baseline): 0.2634
95% CI: [nan, nan]
p-value (raw): 2.743e-221
p-value (adjusted): 2.743e-221
Effect size h: 0.5406 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.6638
metric(no_kb): 0.2885
Diff (model - baseline): 0.3752
95% C

### Combined datasets

In [23]:
# Union of both datasets is treated according to the class distribution in the original datasets and according to the overall dataset size, thus decoding matters more
res_union_weighted = wrap_model_comparison_union(df_union, N_neg_total_by_dataset)
print_model_comparison_results(res_union_weighted)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.4707
metric(no_kb): 0.5604
Diff (model - baseline): -0.0897
95% CI: [-0.1503, -0.0373]
p-value (raw): 0.0002
p-value (adjusted): 0.0002
Effect size h: -0.1797 (95% CI: [-0.3040, -0.0748])

Model: tax
metric(tax): 0.3814
metric(no_kb): 0.5604
Diff (model - baseline): -0.1791
95% CI: [-0.2480, -0.1208]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.3607 (95% CI: [-0.5019, -0.2438])

Model: tax_ex
metric(tax_ex): 0.4388
metric(no_kb): 0.5604
Diff (model - baseline): -0.1216
95% CI: [-0.1874, -0.0648]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.2438 (95% CI: [-0.3782, -0.1300])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.5988
metric(no_kb): 0.3619
Diff (model - baseline): 0.2368
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: 0.4786 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.6902
metric(no_kb): 0.3619
Diff (model - baseline): 0.3282
95% CI: [nan, nan]
p-value (raw

In [24]:
# Union of both datasets is treated according to the class distribution in the original datasets but both datasets weightes equally

res_union_dataset_balanced = combine_model_comparison_outputs(
    {
        "B": res_b_weighted,
        "D": res_d_weighted,
    },
    dataset_weights={
        "B": 0.5,
        "D": 0.5,
    },
)


In [25]:
print_model_comparison_results(res_union_dataset_balanced)


===== PRECISION vs no_kb =====

Model: ihra
metric(ihra): 0.4780
metric(no_kb): 0.5723
Diff (model - baseline): -0.0944
95% CI: [nan, nan]
p-value (raw): 0.2654
p-value (adjusted): 0.2654
Effect size h: -0.1918 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.4068
metric(no_kb): 0.5723
Diff (model - baseline): -0.1655
95% CI: [nan, nan]
p-value (raw): 0
p-value (adjusted): 0
Effect size h: -0.3371 (95% CI: [nan, nan])

Model: tax_ex
metric(tax_ex): 0.4505
metric(no_kb): 0.5723
Diff (model - baseline): -0.1219
95% CI: [nan, nan]
p-value (raw): 0.0454
p-value (adjusted): 0.0908
Effect size h: -0.2472 (95% CI: [nan, nan])

===== RECALL vs no_kb =====

Model: ihra
metric(ihra): 0.6129
metric(no_kb): 0.3840
Diff (model - baseline): 0.2288
95% CI: [nan, nan]
p-value (raw): 1.778e-105
p-value (adjusted): 3.555e-105
Effect size h: 0.4683 (95% CI: [nan, nan])

Model: tax
metric(tax): 0.6981
metric(no_kb): 0.3840
Diff (model - baseline): 0.3141
95% CI: [nan, nan]
p-value (raw): 7.758e-140
p-valu

In [26]:
res_union_dataset_balanced

[('precision',
  {'ihra': {'metric_A': 0.4779619912895723,
    'metric_B': 0.5723479618938958,
    'diff': -0.09438597060432352,
    'effect_size_h': np.float64(-0.19176996682932035),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.2654),
    'p_value_adj': np.float64(0.2654)},
   'tax': {'metric_A': 0.40681097453871573,
    'metric_B': 0.5723479618938958,
    'diff': -0.16553698735518008,
    'effect_size_h': np.float64(-0.3371467325393043),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value': np.float64(0.0),
    'p_value_adj': np.float64(0.0)},
   'tax_ex': {'metric_A': 0.45049528906025693,
    'metric_B': 0.5723479618938958,
    'diff': -0.12185267283363888,
    'effect_size_h': np.float64(-0.24715441382450998),
    'ci_low': nan,
    'ci_high': nan,
    'effect_size_h_ci_low': nan,
    'effect_size_h_ci_high': nan,
    'p_value':

In [27]:
for metric_name, metric_res in res_b_weighted:
    print("\n", metric_name)
    for model_name, r in metric_res.items():
        print(
            model_name,
            "metric_A =", r["metric_A"],
            "metric_B =", r["metric_B"],
        )


 precision
ihra metric_A = 0.5130167763041983 metric_B = 0.660223987100809
tax metric_A = 0.4792989346251757 metric_B = 0.660223987100809
tax_ex metric_A = 0.49562490919401864 metric_B = 0.660223987100809

 recall
ihra metric_A = 0.673842841765339 metric_B = 0.47954790096878364
tax metric_A = 0.732508073196986 metric_B = 0.47954790096878364
tax_ex metric_A = 0.6829924650161464 metric_B = 0.47954790096878364

 f1
ihra metric_A = 0.5825333967978349 metric_B = 0.5555656013233808
tax metric_A = 0.5794492634906098 metric_B = 0.5555656013233808
tax_ex metric_A = 0.5744155581970327 metric_B = 0.5555656013233808


In [28]:
df_B = extract_metric_values(res_b_weighted, "B")
df_D = extract_metric_values(res_d_weighted, "D")
df_union = extract_metric_values(res_union_weighted, "union")
df_union_bal = extract_metric_values(res_union_dataset_balanced, "union_dataset_balanced")

df_metrics = pd.concat(
    [df_B, df_D, df_union, df_union_bal],
    ignore_index=True,
)

In [29]:
df_metrics

,setting,metric,model,value
0,B,precision,no_kb,0.660224
1,B,precision,ihra,0.513017
2,B,precision,tax,0.479299
3,B,precision,tax_ex,0.495625
4,B,recall,no_kb,0.479548
5,B,recall,ihra,0.673843
6,B,recall,tax,0.732508
7,B,recall,tax_ex,0.682992
8,B,f1,no_kb,0.555566
9,B,f1,ihra,0.582533


##  Prediction overlap between models

In [29]:
def intersection_over_union(y_preds, label=1):
    y_preds = [np.asarray(y) for y in y_preds]
    intersection = np.logical_and.reduce([y == label for y in y_preds])
    union = np.logical_or.reduce([y == label for y in y_preds])
    if np.sum(union) == 0:
        return np.nan  # avoid division by zero
    return np.round(np.sum(intersection) / np.sum(union), 2)

### Bloomington

In [30]:
# all 4
intersection_over_union([df_b[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.32)

In [31]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.36
IoU for classification_no_kb_cleaned vs classification_tax: 0.37
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.36
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.87
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.87
IoU for classification_tax vs classification_tax_ex: 0.86


#### Positive samples only

In [32]:
intersection_over_union([
    df_b[df_b.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.36)

In [33]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_b[df_b.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_b[df_b.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.4
IoU for classification_no_kb_cleaned vs classification_tax: 0.41
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.4
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.89
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.89
IoU for classification_tax vs classification_tax_ex: 0.89


### Decoding

In [34]:
# all 4
intersection_over_union([df_d[c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.18)

In [35]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.22
IoU for classification_no_kb_cleaned vs classification_tax: 0.24
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.23
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.8
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.81
IoU for classification_tax vs classification_tax_ex: 0.82


#### Positive samples only

In [36]:
# all 5
intersection_over_union([df_d[df_d.label==1][c].map(lambda x: 1 if x=="Yes" else 0) for c in CLASS_COLS])

np.float64(0.19)

In [37]:
for i,j in combinations(CLASS_COLS,2):
    iou = intersection_over_union([
    df_d[df_d.label==1][i].map(lambda x: 1 if x=="Yes" else 0),
    df_d[df_d.label==1][j].map(lambda x: 1 if x=="Yes" else 0)
    ])
    print(f"IoU for {i} vs {j}: {iou}")

IoU for classification_no_kb_cleaned vs classification_ihra_explanation_cleaned: 0.23
IoU for classification_no_kb_cleaned vs classification_tax: 0.26
IoU for classification_no_kb_cleaned vs classification_tax_ex: 0.24
IoU for classification_ihra_explanation_cleaned vs classification_tax: 0.81
IoU for classification_ihra_explanation_cleaned vs classification_tax_ex: 0.83
IoU for classification_tax vs classification_tax_ex: 0.83
